# DDIM Sampling Step（DDIM 采样步）

**难度：** Medium

**函数签名：** `ddim_step(x_t, noise_pred, alpha_bar_t, alpha_bar_prev) -> Tensor`

**参数：**
- `x_t` — 当前含噪样本 (B, ...)
- `noise_pred` — 模型预测的噪声 ε_θ(x_t, t)
- `alpha_bar_t` — 当前步累积噪声调度 ᾱ_t
- `alpha_bar_prev` — 上一步累积噪声调度 ᾱ_{t-1}

**返回：** x_{t-1}

**公式：**
1. `x0_pred = (x_t - √(1-ᾱ_t)·ε) / √ᾱ_t`
2. `noise_direction = √(1-ᾱ_{t-1})·ε`
3. `x_{t-1} = √ᾱ_{t-1}·x0_pred + noise_direction`

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def ddim_step(x_t, noise_pred, alpha_bar_t, alpha_bar_prev):
    # ---- Step 1: 从含噪样本预测干净样本 x0 ----
    # 扩散前向过程: x_t = √ᾱ_t · x0 + √(1-ᾱ_t) · ε
    # 反解 x0: x0 = (x_t - √(1-ᾱ_t) · ε) / √ᾱ_t
    # 这里 ε 用模型预测值 noise_pred 代替
    x0_pred = (x_t - (1 - alpha_bar_t) ** 0.5 * noise_pred) / (alpha_bar_t ** 0.5)

    # ---- Step 2: 计算指向 x_t 的噪声方向 ----
    # DDIM 的关键: 不注入新的随机噪声，而是用 x_t 的方向
    # noise_direction = √(1-ᾱ_{t-1}) · ε
    # 这决定了从 x0_pred 向噪声方向偏移多少
    noise_direction = (1 - alpha_bar_prev) ** 0.5 * noise_pred

    # ---- Step 3: 计算上一步样本 ----
    # x_{t-1} = √ᾱ_{t-1} · x0_pred + √(1-ᾱ_{t-1}) · ε
    # 这是 DDIM 的确定性采样公式
    # 注意: 与 DDPM 不同，DDIM 不加随机噪声，所以采样是确定性的
    # ᾱ_{t-1} > ᾱ_t（上一步噪声更少），所以 x_{t-1} 更接近 x0
    x_prev = (alpha_bar_prev ** 0.5) * x0_pred + noise_direction
    return x_prev

In [ ]:
torch.manual_seed(42)

# Simple linear alpha_bar schedule for demo
T = 5
alpha_bars = torch.linspace(0.99, 0.01, T + 1)  # index 0..T

x_clean = torch.tensor([1.0, -1.0, 0.5])  # target signal
noise   = torch.randn_like(x_clean)

# Start from pure noise at step T
ab_T = alpha_bars[T]
x_t  = ab_T ** 0.5 * x_clean + (1 - ab_T) ** 0.5 * noise

print(f"{'Step':>4}  {'alpha_bar_t':>12}  {'x_t (first elem)':>18}")
print("-" * 42)
for step in range(T, 0, -1):
    ab_t    = alpha_bars[step]
    ab_prev = alpha_bars[step - 1]
    # Oracle noise prediction for demo
    noise_pred = (x_t - ab_t ** 0.5 * x_clean) / (1 - ab_t) ** 0.5
    x_t = ddim_step(x_t, noise_pred, ab_t, ab_prev)
    print(f"{step:>4}  {ab_prev.item():>12.4f}  {x_t[0].item():>18.4f}")

print(f"\nFinal x vs clean: {x_t.tolist()}  vs  {x_clean.tolist()}")

In [ ]:
from torch_judge import check
check("ddim_step")

## 📝 核心思路总结

1. **DDIM = 确定性采样**：不注入随机噪声，用 x_t 的方向替代
2. **两步公式**：先预测 x0，再沿噪声方向插值
3. **ᾱ 控制信噪比**：ᾱ 越大信号越强，ᾱ 越小噪声越强
4. **与 DDPM 的区别**：DDPM 加随机噪声，DDIM 不加